In [3]:
# Point 1: File paths aur image handling ke liye
import os
from PIL import Image

# Point 2: PyTorch aur layers import karna
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

# Point 3: Har baar ek jaisa result aane ke liye seed set karna
torch.manual_seed(42)

# Point 4: CPU ya GPU set karna
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device ready: {device}")

Device ready: cpu


In [45]:
# Point 1: Saari fingerprint files dhoondhna
all_files = [os.path.join(r, f) for r, d, fs in os.walk('.') for f in fs if f.lower().endswith('.bmp')]

# Point 2: Demo ke liye 50 persons benchmark select karna
NUM_CLASSES = 50

# Point 3: Images ko PyTorch format me badalna
class MiniDataset(Dataset):
    def __init__(self, file_list):
        self.samples = []
        for f in file_list:
            try:
                pid = int(os.path.basename(f).split('__')[0])
                if 1 <= pid <= NUM_CLASSES:
                    self.samples.append((f, pid - 1))
            except:
                pass
        self.tf = transforms.Compose([
            transforms.Resize((96, 96)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]
        return self.tf(Image.open(path).convert('L')), label

# Point 4: 85% Train aur 15% Test split banana
data = MiniDataset(all_files)
train_len = int(0.85 * len(data))
train_set, test_set = torch.utils.data.random_split(data, [train_len, len(data) - train_len])

# Point 5: Batches ready karna
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)
print(f"Total Images: {len(data)} | Train: {len(train_set)} | Test: {len(test_set)}")

Total Images: 4554 | Train: 3870 | Test: 684


In [4]:
# Point 1: Simple 3-Layer CNN Architecture
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 12 * 12, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, NUM_CLASSES)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

# Point 2: Model, Loss aur Optimizer initialize karna
model = SmallCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
print("Chhota CNN Model Ready!")

Chhota CNN Model Ready!


In [47]:
# Point 1: Training start karna
epochs = 5
for epoch in range(epochs):
    model.train()
    correct, total, loss_sum = 0, 0, 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, lbls)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct += (outs.argmax(1) == lbls).sum().item()
        total += lbls.size(0)
    
    # Point 2: Test accuracy check karna
    model.eval()
    t_correct, t_total = 0, 0
    with torch.no_grad():
        for t_imgs, t_lbls in test_loader:
            t_imgs, t_lbls = t_imgs.to(device), t_lbls.to(device)
            t_correct += (model(t_imgs).argmax(1) == t_lbls).sum().item()
            t_total += t_lbls.size(0)
            
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {loss_sum/total:.4f} | Train Acc: {correct/total*100:.2f}% | Test Acc: {t_correct/t_total*100:.2f}%")

# Point 3: Model save karna
torch.save(model.state_dict(), "fingerprint_cnn_model.pth")
print("\nSuccess! Model save ho gaya.")

Epoch [1/5] | Loss: 4.2510 | Train Acc: 3.59% | Test Acc: 8.33%
Epoch [2/5] | Loss: 3.4794 | Train Acc: 9.82% | Test Acc: 17.98%
Epoch [3/5] | Loss: 3.0929 | Train Acc: 15.01% | Test Acc: 23.39%
Epoch [4/5] | Loss: 2.6740 | Train Acc: 22.84% | Test Acc: 37.57%
Epoch [5/5] | Loss: 2.3446 | Train Acc: 30.98% | Test Acc: 38.89%

Success! Model save ho gaya.


In [48]:
# Point 1: 5 epochs aur aage train karna
epochs_extra = 5

for epoch in range(epochs_extra):
    model.train()
    correct, total, loss_sum = 0, 0, 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, lbls)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct += (outs.argmax(1) == lbls).sum().item()
        total += lbls.size(0)
        
    model.eval()
    t_correct, t_total = 0, 0
    with torch.no_grad():
        for t_imgs, t_lbls in test_loader:
            t_imgs, t_lbls = t_imgs.to(device), t_lbls.to(device)
            t_correct += (model(t_imgs).argmax(1) == t_lbls).sum().item()
            t_total += t_lbls.size(0)
            
    print(f"Bonus Epoch [{epoch+1}/{epochs_extra}] | Loss: {loss_sum/total:.4f} | Train Acc: {correct/total*100:.2f}% | Test Acc: {t_correct/t_total*100:.2f}%")

# Point 2: Boosted model wapas save karna
torch.save(model.state_dict(), "fingerprint_cnn_model.pth")
print("\nSuccess! High-accuracy model save ho gaya.")

Bonus Epoch [1/5] | Loss: 2.0771 | Train Acc: 37.98% | Test Acc: 40.79%
Bonus Epoch [2/5] | Loss: 1.8203 | Train Acc: 43.80% | Test Acc: 64.18%
Bonus Epoch [3/5] | Loss: 1.6319 | Train Acc: 49.69% | Test Acc: 69.44%
Bonus Epoch [4/5] | Loss: 1.4853 | Train Acc: 53.54% | Test Acc: 72.08%
Bonus Epoch [5/5] | Loss: 1.3512 | Train Acc: 56.59% | Test Acc: 77.92%

Success! High-accuracy model save ho gaya.


In [5]:
import os
import streamlit as st
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image

NUM_CLASSES = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Wahi SmallCNN architecture jo training me save hui
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 12 * 12, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, NUM_CLASSES)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

transform = transforms.Compose([
    transforms.Resize((96, 96)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

@st.cache_resource
def load_trained_model():
    net = SmallCNN().to(device)
    if os.path.exists("fingerprint_cnn_model.pth"):
        net.load_state_dict(torch.load("fingerprint_cnn_model.pth", map_location=device))
        net.eval()
        return net
    return None

model = load_trained_model()

st.set_page_config(page_title="Biometric Fingerprint System", page_icon="🔍")
st.title("🔍 Biometric Fingerprint Recognition")
st.write("Person 1 se 50 ke beech ki koi fingerprint image upload karein.")

if model is None:
    st.error("Model file nahi mili! Check karein ki 'fingerprint_cnn_model.pth' folder me ho.")
else:
    uploaded_file = st.file_uploader("Select Fingerprint (.BMP)", type=["bmp", "png", "jpg", "jpeg"])
    if uploaded_file is not None:
        image = Image.open(uploaded_file).convert("L")
        col1, col2 = st.columns(2)
        
        with col1:
            st.image(image, caption="Uploaded Fingerprint", width=220)
            
        with col2:
            if st.button("Identify Person"):
                img_tensor = transform(image).unsqueeze(0).to(device)
                with torch.no_grad():
                    output = model(img_tensor)
                    probs = torch.nn.functional.softmax(output, dim=1)
                    conf, pred = torch.max(probs, 1)
                    
                person_id = pred.item() + 1
                conf_pct = conf.item() * 100
                
                st.success(f"Match Found: **Person_{person_id}**")
                st.metric(label="Matching Confidence", value=f"{conf_pct:.2f}%")

2026-10-11 10:25:20.903 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-11 10:25:20.904 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-11 10:25:20.904 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-11 10:25:20.905 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-11 10:25:20.906 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-11 10:25:20.908 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-11 10:25:20.909 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-10-11 10:25:20.911 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [7]:
import os

# Point 1: Folder scan karke Person 1 se 50 ki files dhundhna
selected_files = []
for root, dirs, files in os.walk('.'):
    for f in files:
        if f.lower().endswith('.bmp'):
            try:
                pid = int(f.split('__')[0])
                if 1 <= pid <= 50:
                    selected_files.append((pid, os.path.join(root, f)))
            except:
                pass

# Point 2: Total count aur paths display karna
print(f"Total Selected Images (Person 1 se 50): {len(selected_files)}\n")

print("--- Kuch Initial Samples ---")
for pid, path in selected_files[:5]:
    print(f"Person {pid}: {path}")

print("...\n--- Kuch Last Samples ---")
for pid, path in selected_files[-5:]:
    print(f"Person {pid}: {path}")
    

Total Selected Images (Person 1 se 50): 4554

--- Kuch Initial Samples ---
Person 10: .\dataset\SOCOFing\SOCOFing\Altered\Altered-Easy\10__M\10__M_Left_index_finger_CR.BMP
Person 10: .\dataset\SOCOFing\SOCOFing\Altered\Altered-Easy\10__M\10__M_Left_index_finger_Obl.BMP
Person 10: .\dataset\SOCOFing\SOCOFing\Altered\Altered-Easy\10__M\10__M_Left_index_finger_Zcut.BMP
Person 10: .\dataset\SOCOFing\SOCOFing\Altered\Altered-Easy\10__M\10__M_Left_little_finger_CR.BMP
Person 10: .\dataset\SOCOFing\SOCOFing\Altered\Altered-Easy\10__M\10__M_Left_little_finger_Obl.BMP
...
--- Kuch Last Samples ---
Person 9: .\dataset\SOCOFing\SOCOFing\Real\9__M\9__M_Right_index_finger.BMP
Person 9: .\dataset\SOCOFing\SOCOFing\Real\9__M\9__M_Right_little_finger.BMP
Person 9: .\dataset\SOCOFing\SOCOFing\Real\9__M\9__M_Right_middle_finger.BMP
Person 9: .\dataset\SOCOFing\SOCOFing\Real\9__M\9__M_Right_ring_finger.BMP
Person 9: .\dataset\SOCOFing\SOCOFing\Real\9__M\9__M_Right_thumb_finger.BMP


In [8]:
import os
import shutil

# Destination folder create karna
output_folder = "Selected_50_Persons"
os.makedirs(output_folder, exist_ok=True)

# 4554 files ko naye folder me copy karna
for pid, src_path in selected_files:
    filename = os.path.basename(src_path)
    dst_path = os.path.join(output_folder, filename)
    shutil.copy2(src_path, dst_path)

print(f"Done! Saari {len(selected_files)} files '{output_folder}' folder me save ho chuki hain.")

Done! Saari 4554 files 'Selected_50_Persons' folder me save ho chuki hain.


In [9]:
import os
from PIL import Image
import torch
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device ready: {device}")

Device ready: cpu


In [ ]:
#README FILE KE DATA SPLIT KRNE KE LIE

all_files = [os.path.join(r, f) for r, d, fs in os.walk('.') for f in fs if f.lower().endswith('.bmp')]
NUM_CLASSES = 50

class MiniDataset(Dataset):
    def __init__(self, file_list):
        self.samples = []
        for f in file_list:
            try:
                pid = int(os.path.basename(f).split('__')[0])
                if 1 <= pid <= NUM_CLASSES:
                    self.samples.append((f, pid - 1))
            except:
                pass
        self.tf = transforms.Compose([
            transforms.Resize((96, 96)),
            transforms.ToTensor(),
            transforms.Normalize((0.5,), (0.5,))
        ])
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]
        return self.tf(Image.open(path).convert('L')), label

In [ ]:
# Point 4: 85% Train aur 15% Test split banana
data = MiniDataset(all_files)
train_len = int(0.85 * len(data))
train_set, test_set = torch.utils.data.random_split(data, [train_len, len(data) - train_len])

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

In [ ]:
# Point 4: 85% Train aur 15% Test split banana
data = MiniDataset(all_files)
train_len = int(0.85 * len(data))
train_set, test_set = torch.utils.data.random_split(data, [train_len, len(data) - train_len])

train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

In [ ]:
# Point 1: Simple 3-Layer CNN Architecture
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2, 2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2, 2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 12 * 12, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, NUM_CLASSES)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

In [12]:
# Point 2: Model, Loss aur Optimizer initialize karna
model = SmallCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 5
for epoch in range(epochs):
    model.train()
    correct, total, loss_sum = 0, 0, 0.0
    for imgs, lbls in train_loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        optimizer.zero_grad()
        outs = model(imgs)
        loss = criterion(outs, lbls)
        loss.backward()
        optimizer.step()
        loss_sum += loss.item() * imgs.size(0)
        correct += (outs.argmax(1) == lbls).sum().item()
        total += lbls.size(0)

    model.eval()
    t_correct, t_total = 0, 0
    with torch.no_grad():
        for t_imgs, t_lbls in test_loader:
            t_imgs, t_lbls = t_imgs.to(device), t_lbls.to(device)
            t_correct += (model(t_imgs).argmax(1) == t_lbls).sum().item()
            t_total += t_lbls.size(0)

    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {loss_sum/total:.4f} | Train Acc: {correct/total*100:.2f}% | Test Acc: {t_correct/t_total*100:.2f}%")

torch.save(model.state_dict(), "fingerprint_cnn_model.pth")

Epoch [1/5] | Loss: 4.3959 | Train Acc: 6.17% | Test Acc: 9.98%
Epoch [2/5] | Loss: 3.2365 | Train Acc: 14.33% | Test Acc: 27.39%
Epoch [3/5] | Loss: 2.6194 | Train Acc: 25.97% | Test Acc: 45.11%
Epoch [4/5] | Loss: 2.0187 | Train Acc: 40.84% | Test Acc: 60.39%
Epoch [5/5] | Loss: 1.5182 | Train Acc: 54.61% | Test Acc: 70.77%
